# Fase 3 — TCN temporal sobre el catálogo USGS

Este notebook modela la dimensión temporal de la sismicidad con una red convolucional temporal (TCN) sobre características globales agregadas de la región. El objetivo es comprobar si la evolución temporal de la actividad, por sí sola, permite anticipar mainshocks, antes de pasar a la formulación espacial de los notebooks 04 y 05.

| Aspecto | Valor |
|---|---|
| Datos | Catálogo USGS (California + Alaska) |
| Periodo | 2000–2024 |
| Magnitud de completitud | M ≥ 2.5 |
| Objetivo | mainshock M ≥ 4.5 en cualquiera de las dos regiones en los próximos 14 días |
| Características | 10 canales globales (5 por región × 2 regiones) por día |
| Referencias | persistencia, regresión logística sobre características agregadas |
| Métricas | AUC-ROC, multi-semilla (5 semillas), t-test frente a la regresión logística |

El objetivo se define sobre mainshocks independientes (tras el declustering de Gardner–Knopoff). La ventana de predicción es de 14 días: con las dos regiones combinadas, una ventana de 30 días deja el objetivo casi siempre positivo.

## Estructura del notebook

1. Carga y exploración del catálogo.
2. Características globales por región.
3. Declustering y definición del objetivo.
4. Entrenamiento de la TCN.
5. Baselines y comparación.
6. Evaluación final.

In [ ]:
# Entorno + NB_TAG (derivado automaticamente del NOMBRE de este notebook)
import os

try:
    from google.colab import drive

    # Si el import funciona, el entorno es Colab
    print("Entorno Colab detectado: Montando Google Drive...")
    drive.mount('/content/drive')

    import sys
    sys.path.append("/content/drive/MyDrive/VIU/TFM/code")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/results")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/USGS")

    # Nombre del notebook en Colab -> NB_TAG = primera parte antes del '_'
    from google.colab import _message
    _meta = _message.blocking_request("get_ipynb")["ipynb"]["metadata"]
    _nb_name = _meta.get("colab", {}).get("name", "03.ipynb")
    NB_TAG = os.path.splitext(_nb_name)[0].split("_")[0]

except ImportError:
    # Entorno local (VS Code / Jupyter): nombre del fichero -> NB_TAG
    try:
        NB_TAG = os.path.splitext(os.path.basename(__vsc_ipynb_file__))[0].split("_")[0]
    except NameError:
        NB_TAG = "03"

print(f"NB_TAG = {NB_TAG}")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append(os.path.abspath(""))

!pip install -q seisbench

# Reproducibilidad (semillas fijas)
SEED = 42
np.random.seed(SEED)

## 1. Carga y exploración del catálogo

In [ ]:
# --- Configuración + descarga del catálogo USGS (California + Alaska, 2000–2024) ---
import urllib.parse
import urllib.error

REGIONS = [
    {"name": "california", "minlatitude":  32.0, "maxlatitude":  42.0,
                           "minlongitude": -124.0, "maxlongitude": -114.0},
    {"name": "alaska",     "minlatitude":  51.0, "maxlatitude":  72.0,
                           "minlongitude": -180.0, "maxlongitude": -130.0},
]
REGIONS_TAG       = "+".join(r["name"] for r in REGIONS)
STARTTIME         = "2000-01-01"
ENDTIME           = "2024-12-31"
MIN_MAG_CATALOGO  = 2.5
MIN_MAG_TARGET    = 4.5
PREDICTION_WINDOW = 14
LOOKBACK          = 60

# --- Rutas ---
RESULTS_DIR = os.path.join("..", "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
USGS_DIR    = os.path.join("..", "USGS")
for d in (RESULTS_DIR, FIGURES_DIR, USGS_DIR):
    os.makedirs(d, exist_ok = True)


def fetch_region_catalog(region):
    """Carga del cache o descarga el catálogo USGS para una región."""
    path = os.path.join(USGS_DIR,
        f"{region['name']}_{STARTTIME[:4]}_{ENDTIME[:4]}_M{MIN_MAG_CATALOGO}.csv")
    if os.path.exists(path):
        print(f"  [{region['name']}] caché: {path}")
        df = pd.read_csv(path)
        df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
        return df

    print(f"  [{region['name']}] descargando desde USGS FDSN...")
    BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    fechas_start = pd.date_range(STARTTIME, ENDTIME, freq = "MS")
    fechas_end   = list(fechas_start[1:]) + [pd.to_datetime(ENDTIME) + pd.Timedelta(days = 1)]

    dfs = []
    for inicio, fin in tqdm(list(zip(fechas_start, fechas_end)),
                            desc = f"  USGS {region['name']}"):
        s, e = inicio.strftime("%Y-%m-%d"), fin.strftime("%Y-%m-%d")
        params = {
            "format": "csv", "starttime": s, "endtime": e,
            "minmagnitude": MIN_MAG_CATALOGO,
            "minlatitude":  region["minlatitude"],  "maxlatitude":  region["maxlatitude"],
            "minlongitude": region["minlongitude"], "maxlongitude": region["maxlongitude"],
            "orderby": "time-asc",
        }
        url = BASE_URL + "?" + urllib.parse.urlencode(params)
        try:
            df_chunk = pd.read_csv(url)
        except (pd.errors.EmptyDataError, urllib.error.HTTPError):
            df_chunk = pd.DataFrame()
        dfs.append(df_chunk)

    df = pd.concat(dfs, ignore_index = True)
    df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
    df = (df.drop_duplicates(subset = ["id"]).sort_values("time").reset_index(drop = True))
    df.to_csv(path, index = False)
    print(f"  [{region['name']}] guardado: {path}")
    return df


print(f"Cargando {len(REGIONS)} región(es)...")
dfs_per_region = []
for region in REGIONS:
    df_r = fetch_region_catalog(region)
    df_r["region"] = region["name"]
    dfs_per_region.append(df_r)
    print(f"    └─ {region['name']}: {len(df_r):,} eventos M≥{MIN_MAG_CATALOGO}, "
          f"{(df_r['mag'] >= MIN_MAG_TARGET).sum()} M≥{MIN_MAG_TARGET}")

df_cat = pd.concat(dfs_per_region, ignore_index = True).sort_values("time").reset_index(drop = True)
df_cat["date"] = df_cat["time"].dt.floor("D")

n_years = (pd.to_datetime(ENDTIME) - pd.to_datetime(STARTTIME)).days / 365.25
print(f"\n=== Configuración ===")
print(f"Regiones:          {REGIONS_TAG}  ({len(REGIONS)} bbox)")
print(f"Ventana temporal:  {STARTTIME} → {ENDTIME}  ({n_years:.1f} años)")
print(f"M catálogo ≥ {MIN_MAG_CATALOGO}  |  M objetivo ≥ {MIN_MAG_TARGET}")
print(f"PREDICTION_WINDOW: {PREDICTION_WINDOW} días  |  LOOKBACK: {LOOKBACK} días")
print()
print(f"=== Catálogo combinado ===")
print(f"Total eventos:     {len(df_cat):,}")
print(f"Rango temporal:    {df_cat['time'].min()} → {df_cat['time'].max()}")
print(f"Eventos M ≥ {MIN_MAG_TARGET}:  {(df_cat['mag'] >= MIN_MAG_TARGET).sum():,}  "
      f"(~{(df_cat['mag'] >= MIN_MAG_TARGET).sum() / n_years:.1f}/año)")
print()
print(df_cat.groupby("region").agg(
    n_total      = ("mag", "size"),
    n_M_target   = ("mag", lambda m: (m >= MIN_MAG_TARGET).sum()),
).to_string())

In [ ]:
# --- Resumen del catálogo + top 15 eventos significativos ---
df_grandes = df_cat[df_cat["mag"] >= MIN_MAG_TARGET].copy()

print(f"=== Estadísticos generales ===")
print(f"Periodo:                {df_cat['time'].min().date()} → {df_cat['time'].max().date()} ({n_years:.1f} años)")
print(f"Eventos totales:        {len(df_cat):,}")
print(f"Eventos M ≥ {MIN_MAG_TARGET}:        {len(df_grandes):,}  ({len(df_grandes) / n_years:.1f}/año)")
print(f"Magnitud máx:           M = {df_cat['mag'].max():.2f}")
print(f"Profundidad mediana:    {df_cat['depth'].median():.1f} km")
print()
print(f"=== Top 15 eventos M ≥ {MIN_MAG_TARGET} ===")

df_grandes.nlargest(15, "mag")[["time", "latitude", "longitude", "depth", "mag", "region", "place"]]

In [ ]:
# --- Figura EDA: temporal · magnitudes · Gutenberg-Richter · espacial — 300 DPI ---
fig, axs = plt.subplots(2, 2, figsize = (16, 12))

# (a) Distribución temporal por año
ax = axs[0, 0]
df_cat["year"] = df_cat["time"].dt.year
counts_y = df_cat.groupby(["year", "region"]).size().unstack(fill_value = 0)
counts_y.plot(kind = "bar", stacked = True, ax = ax, color = ["steelblue", "darkorange"],
              edgecolor = "black", linewidth = 0.4, alpha = 0.85)
ax.set_xlabel("Año"); ax.set_ylabel("Nº de eventos")
ax.set_title("Distribución temporal por región", fontweight = "bold")
ax.grid(True, alpha = 0.3, axis = "y")

# (b) Histograma de magnitudes (escala log)
ax = axs[0, 1]
mag_bins = np.arange(MIN_MAG_CATALOGO, df_cat["mag"].max() + 0.1, 0.1)
for region_name, color in zip(["california", "alaska"], ["steelblue", "darkorange"]):
    df_r = df_cat[df_cat["region"] == region_name]
    ax.hist(df_r["mag"], bins = mag_bins, color = color, alpha = 0.6, edgecolor = "black",
            linewidth = 0.3, label = region_name)
ax.axvline(MIN_MAG_TARGET, color = "red", linestyle = "--", linewidth = 2, label = f"M ≥ {MIN_MAG_TARGET}")
ax.set_yscale("log"); ax.set_xlabel("Magnitud"); ax.set_ylabel("Nº de eventos (log)")
ax.set_title("Distribución de magnitudes", fontweight = "bold"); ax.legend()

# (c) Ley de Gutenberg-Richter
ax = axs[1, 0]
for region_name, color in zip(["california", "alaska"], ["steelblue", "darkorange"]):
    df_r = df_cat[df_cat["region"] == region_name]
    mag_sorted = np.sort(df_r["mag"].dropna().values)
    N_cum      = np.arange(len(mag_sorted), 0, -1)
    ax.semilogy(mag_sorted, N_cum, color = color, linewidth = 1.5, label = region_name)
    mag_above_mc = mag_sorted[mag_sorted >= MIN_MAG_CATALOGO]
    if len(mag_above_mc) >= 20:
        b = np.log10(np.e) / (np.mean(mag_above_mc) - (MIN_MAG_CATALOGO - 0.05))
        ax.text(0.05, 0.05 + 0.05 * (region_name == "alaska"),
                f"b ({region_name}) = {b:.2f}", transform = ax.transAxes, color = color)
ax.axvline(MIN_MAG_TARGET, color = "red", linestyle = "--", linewidth = 1.5)
ax.set_xlabel("Magnitud M"); ax.set_ylabel("N (M ≥ x)")
ax.set_title("Ley de Gutenberg-Richter por región", fontweight = "bold")
ax.legend(); ax.grid(True, alpha = 0.3)

# (d) Mapa espacial
ax = axs[1, 1]
sc = ax.scatter(df_cat["longitude"], df_cat["latitude"], c = df_cat["mag"],
                cmap = "viridis", s = 2, alpha = 0.4)
ax.scatter(df_grandes["longitude"], df_grandes["latitude"],
           c = "red", s = 60, marker = "*", edgecolor = "black", linewidth = 0.5,
           label = f"M ≥ {MIN_MAG_TARGET}", zorder = 5)
plt.colorbar(sc, ax = ax, label = "Magnitud")
ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")
ax.set_title("Distribución espacial", fontweight = "bold"); ax.legend(loc = "lower left")

plt.suptitle(
    f"EDA Catálogo USGS — {REGIONS_TAG} {STARTTIME[:4]}–{ENDTIME[:4]} (M ≥ {MIN_MAG_CATALOGO})",
    fontsize = 14, fontweight = "bold",
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{NB_TAG}_eda_catalogo_{REGIONS_TAG.replace('+','_')}.png"),
            dpi = 300, bbox_inches = "tight")
plt.show()

## 2. Características globales por región

In [ ]:
# --- Features GLOBALES POR REGIÓN concatenadas (10 canales = 5 × 2 regiones) ---
# Formulación global (sin rejilla) pero distinguiendo región: para cada día t se
# calculan 5 características por región y se concatenan en un vector de 10 canales.

date_range = pd.date_range(STARTTIME, ENDTIME, freq = "D", tz = "UTC")
n_days     = len(date_range)
date_to_idx = {d: i for i, d in enumerate(date_range)}

WINDOW_BVAL = 30

def compute_region_features(df_region, date_range):
    """5 canales globales por día para una región: log_count, log_energy, max_mag, log_count_M35, b_value_30d."""
    df_r = df_region.copy()
    df_r["energy"] = 10 ** (1.5 * df_r["mag"] + 4.8)
    g = df_r.groupby("date")

    counts     = g.size().reindex(date_range, fill_value = 0)
    max_mag    = g["mag"].max().reindex(date_range).fillna(0.0)
    counts_M35 = (df_r[df_r["mag"] >= 3.5].groupby("date").size()
                                          .reindex(date_range, fill_value = 0))
    energy_sum = g["energy"].sum().reindex(date_range, fill_value = 0.0)

    # b-value móvil 30 días por región
    mags_per_day = (df_r[df_r["mag"] >= MIN_MAG_CATALOGO]
                      .groupby("date")["mag"].apply(list)
                      .reindex(date_range)
                      .apply(lambda x: x if isinstance(x, list) else []))
    b_value = np.full(len(date_range), np.nan, dtype = np.float32)
    for i in range(len(date_range)):
        lo = max(0, i - WINDOW_BVAL + 1)
        mags_win = [m for sub in mags_per_day.iloc[lo:i+1] for m in sub]
        if len(mags_win) >= 20:
            b_value[i] = np.log10(np.e) / (np.mean(mags_win) - (MIN_MAG_CATALOGO - 0.05))
    b_value = pd.Series(b_value, index = date_range).ffill().fillna(1.0).values

    return pd.DataFrame({
        "log_count":     np.log1p(counts).astype(np.float32),
        "log_energy":    np.log1p(energy_sum).astype(np.float32),
        "max_mag":       max_mag.astype(np.float32),
        "log_count_M35": np.log1p(counts_M35).astype(np.float32),
        "b_value_30d":   b_value.astype(np.float32),
    }, index = date_range)


# Calcular features por región y concatenar columnas
feature_frames = {}
for region_name in [r["name"] for r in REGIONS]:
    df_region = df_cat[df_cat["region"] == region_name]
    feature_frames[region_name] = compute_region_features(df_region, date_range)
    print(f"  features {region_name:12s}  shape {feature_frames[region_name].shape}")

# Concatenar: cada canal con sufijo de región
features = pd.concat(
    [feature_frames[r["name"]].add_suffix(f"_{r['name']}") for r in REGIONS],
    axis = 1,
)
FEATURE_COLS = list(features.columns)
n_features = len(FEATURE_COLS)

print(f"\nFeatures (n_días, n_canales) = {features.shape}")
print(f"Columnas ({n_features}):")
for col in FEATURE_COLS:
    print(f"  {col}")
print()
print(features.describe().round(3))

## 3. Declustering y definición del objetivo

In [ ]:
# --- Declustering Gardner-Knopoff (1974) sobre el catálogo combinado ---
# Filtra aftershocks/foreshocks dependientes para que el target sean SOLO mainshocks
# independientes (válidos como precursores reales, no réplicas).

def gardner_knopoff_window(M):
    R_km = 10 ** (0.1238 * M + 0.983)
    if M <= 6.5:
        T_days = 10 ** (0.5409 * M - 0.547)
    else:
        T_days = 10 ** (0.032 * M + 2.7389)
    return T_days, R_km

def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    lat1r = np.radians(lat1)
    lat2r = np.radians(lat2_arr)
    dlat  = lat2r - lat1r
    dlon  = np.radians(lon2_arr - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df_target_pool = (df_cat[df_cat["mag"] >= MIN_MAG_TARGET]
                    .sort_values("time").reset_index(drop = True))
times_sec = (df_target_pool["time"] - df_target_pool["time"].iloc[0]).dt.total_seconds().values
lats      = df_target_pool["latitude"].values
lons      = df_target_pool["longitude"].values
mags      = df_target_pool["mag"].values
n         = len(df_target_pool)

is_cluster  = np.zeros(n, dtype = bool)
sort_by_mag = np.argsort(-mags)

for i in sort_by_mag:
    if is_cluster[i]:
        continue
    M_main       = mags[i]
    T_days, R_km = gardner_knopoff_window(M_main)
    T_sec        = T_days * 86400.0
    dt           = np.abs(times_sec - times_sec[i])
    candidates   = np.where((dt <= T_sec) & (dt > 0) & (mags < M_main) & (~is_cluster))[0]
    if len(candidates) == 0:
        continue
    dist        = haversine_km(lats[i], lons[i], lats[candidates], lons[candidates])
    cluster_idx = candidates[dist <= R_km]
    is_cluster[cluster_idx] = True

df_targets = df_target_pool[~is_cluster].copy()
df_targets["date"] = df_targets["time"].dt.floor("D")

print(f"=== Declustering Gardner-Knopoff (1974) — catálogo combinado ===")
print(f"Eventos M ≥ {MIN_MAG_TARGET} originales: {n}")
print(f"Mainshocks independientes:              {len(df_targets)}  "
      f"({100 * len(df_targets) / n:.1f}%)")
print(f"Aftershocks/foreshocks eliminados:      {n - len(df_targets)}")
print(f"Tasa anual de mainshocks:               {len(df_targets) / n_years:.2f}/año")
print()
print("Por región:")
print(df_targets["region"].value_counts().to_string())

In [ ]:
# --- Target binario GLOBAL: "¿hay mainshock en próx. T días en CUALQUIER región?" ---
# Con PREDICTION_WINDOW=14 (más corto que los 30d del 03) el ratio positivo baja.
# El modelo predice un único logit: 1 si hay mainshock M≥4.5 en CA o AK
# en los próximos 14 días, 0 si no.

region_names = [r["name"] for r in REGIONS]

# Días con mainshock independiente en CUALQUIER región
is_target_day = np.zeros(n_days, dtype = bool)
for r_idx, r_name in enumerate(region_names):
    df_r = df_targets[df_targets["region"] == r_name]
    if len(df_r) > 0:
        g = df_r.groupby("date").size().reindex(date_range, fill_value = 0)
        is_target_day |= (g > 0).values

# y_full[t] = 1 si en (t+1, ..., t+PREDICTION_WINDOW) hay algún mainshock independiente
y_full = np.zeros(n_days, dtype = np.int32)
for t in range(n_days - PREDICTION_WINDOW):
    if is_target_day[t + 1 : t + 1 + PREDICTION_WINDOW].any():
        y_full[t] = 1

target_days = date_range[is_target_day].values

n_pos = y_full.sum()
print(f"=== Target binario GLOBAL (post-declustering) ===")
print(f"Ventana de predicción: {PREDICTION_WINDOW} días")
print(f"Output del modelo:     1 logit (global CA+AK)")
print()
print(f"target=1: {n_pos:>5} días ({100*y_full.mean():.1f}%)")
print(f"target=0: {n_days - n_pos:>5} días ({100*(1-y_full.mean()):.1f}%)")
print(f"Mainshocks únicos (CA+AK): {is_target_day.sum()}")

In [ ]:
# --- Figura: inspección visual de features vs target GLOBAL (300 DPI) ---
fig, axs = plt.subplots(4, 1, figsize = (16, 12), sharex = True)

t_idx = features.index

# (a) log_count por región
ax = axs[0]
for region_name, color in zip(["california", "alaska"], ["steelblue", "darkorange"]):
    ax.plot(t_idx, features[f"log_count_{region_name}"], color = color, linewidth = 0.5,
            alpha = 0.7, label = region_name)
ax.set_ylabel("log(1 + N)"); ax.set_title("log-sismicidad diaria por región", fontweight = "bold")
ax.legend(loc = "upper right"); ax.grid(True, alpha = 0.3)

# (b) log_energy por región
ax = axs[1]
for region_name, color in zip(["california", "alaska"], ["steelblue", "darkorange"]):
    ax.plot(t_idx, features[f"log_energy_{region_name}"], color = color,
            marker = ".", markersize = 1.0, linestyle = "none", alpha = 0.5, label = region_name)
ax.set_ylabel("log(1 + E)"); ax.set_title("Energía sísmica acumulada (Kanamori)", fontweight = "bold")
ax.legend(loc = "upper right"); ax.grid(True, alpha = 0.3)

# (c) b-value por región
ax = axs[2]
for region_name, color in zip(["california", "alaska"], ["purple", "olive"]):
    ax.plot(t_idx, features[f"b_value_30d_{region_name}"], color = color, linewidth = 0.7,
            alpha = 0.85, label = region_name)
ax.axhline(1.0, color = "gray", linestyle = "--", alpha = 0.5)
ax.set_ylabel("b-value"); ax.set_title("b-value móvil 30 días por región", fontweight = "bold")
ax.legend(loc = "upper right"); ax.grid(True, alpha = 0.3)

# (d) Target global (CA+AK)
ax = axs[3]
ax.fill_between(t_idx, 0, y_full, color = "crimson", alpha = 0.35,
                label = f"target = 1 ({100*y_full.mean():.1f}%)")
for d in target_days:
    ax.axvline(d, color = "darkred", linewidth = 0.4, alpha = 0.5)
ax.set_ylim(-0.05, 1.1); ax.set_ylabel("Target global")
ax.set_xlabel("Fecha")
ax.set_title(f"Target global CA+AK — mainshocks independientes (T={PREDICTION_WINDOW}d)",
             fontweight = "bold")
ax.legend(loc = "upper right"); ax.grid(True, alpha = 0.3)

plt.suptitle(f"EDA Features y target — {REGIONS_TAG} {STARTTIME[:4]}–{ENDTIME[:4]}",
             fontsize = 14, fontweight = "bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,
            f"{NB_TAG}_features_target_M{MIN_MAG_TARGET}_T{PREDICTION_WINDOW}d.png"),
            dpi = 300, bbox_inches = "tight")
plt.show()

## 4. Entrenamiento de la TCN

In [ ]:
# --- Ventanas deslizantes + split temporal train/val/test + normalización z-score ---
LOOKBACK   = 60
TRAIN_END  = "2020-12-31"
VAL_END    = "2022-12-31"

X_full = features[FEATURE_COLS].values.astype(np.float32)   # (n_days, 10)
# y_full shape: (n_days,) — binario global CA+AK

valid_t = np.arange(LOOKBACK, n_days - PREDICTION_WINDOW)
X_seq   = np.stack([X_full[t - LOOKBACK : t, :] for t in valid_t])  # (n_samples, 60, 10)
y_seq   = y_full[valid_t]                                            # (n_samples,)
dates_seq = features.index[valid_t]

train_mask = dates_seq <= pd.Timestamp(TRAIN_END, tz = "UTC")
val_mask   = (dates_seq > pd.Timestamp(TRAIN_END, tz = "UTC")) & \
             (dates_seq <= pd.Timestamp(VAL_END, tz = "UTC"))
test_mask  = dates_seq > pd.Timestamp(VAL_END, tz = "UTC")

X_train_raw, y_train = X_seq[train_mask], y_seq[train_mask]
X_val_raw,   y_val   = X_seq[val_mask],   y_seq[val_mask]
X_test_raw,  y_test  = X_seq[test_mask],  y_seq[test_mask]

flat_train = X_train_raw.reshape(-1, X_train_raw.shape[-1])
mean_train = flat_train.mean(axis = 0, keepdims = True)
std_train  = flat_train.std(axis = 0, keepdims = True) + 1e-8

X_train = (X_train_raw - mean_train) / std_train
X_val   = (X_val_raw   - mean_train) / std_train
X_test  = (X_test_raw  - mean_train) / std_train

n_neg_tr     = (y_train == 0).sum()
n_pos_tr     = y_train.sum()
POS_WEIGHT   = float(n_neg_tr / max(n_pos_tr, 1))

print(f"=== Splits temporales ===")
print(f"LOOKBACK={LOOKBACK}  PREDICTION_WINDOW={PREDICTION_WINDOW}  n_features={n_features}")
print(f"Shapes: X_train {X_train.shape}  y_train {y_train.shape}")
print(f"        X_val   {X_val.shape}   y_val   {y_val.shape}")
print(f"        X_test  {X_test.shape}  y_test  {y_test.shape}")
print()
print(f"Train: {dates_seq[train_mask].min().date()} → {dates_seq[train_mask].max().date()} ({len(X_train)} muestras)")
print(f"Val:   {dates_seq[val_mask].min().date()} → {dates_seq[val_mask].max().date()} ({len(X_val)} muestras)")
print(f"Test:  {dates_seq[test_mask].min().date()} → {dates_seq[test_mask].max().date()} ({len(X_test)} muestras)")
print()
print(f"Ratio positivos (train): {y_train.mean()*100:.1f}%  ({n_pos_tr} días / {len(y_train)} total)")
print(f"Ratio positivos (val):   {y_val.mean()*100:.1f}%")
print(f"Ratio positivos (test):  {y_test.mean()*100:.1f}%")
print(f"pos_weight: {POS_WEIGHT:.3f}")

In [ ]:
# --- Definición del TCN (salida única: logit global CA+AK) ---
import importlib
import torch
import utils.modelos
importlib.reload(utils.modelos)
from utils.modelos import TCNClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

NUM_CHANNELS = (32, 32, 64, 64)
KERNEL_SIZE  = 3
DROPOUT      = 0.20

n_layers        = len(NUM_CHANNELS)
receptive_field = 1 + 2 * (KERNEL_SIZE - 1) * (2 ** n_layers - 1)

torch.manual_seed(SEED)
model = TCNClassifier(
    num_inputs   = X_train.shape[-1],   # 10 features (5 × 2 regiones)
    num_channels = NUM_CHANNELS,
    kernel_size  = KERNEL_SIZE,
    dropout      = DROPOUT,
    # num_outputs=1 por defecto → salida escalar (batch,)
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n=== TCNClassifier {NB_TAG} (salida global) ===")
print(f"Inputs:           {X_train.shape[-1]} canales × {LOOKBACK} días")
print(f"Output:           1 logit (global CA+AK, T={PREDICTION_WINDOW}d)")
print(f"Num. canales:     {NUM_CHANNELS}  (RF temporal = {receptive_field} días)")
print(f"Dropout:          {DROPOUT}")
print(f"Parámetros:       {n_params:,}")
print()

# Smoke test
with torch.no_grad():
    x_dummy   = torch.randn(8, LOOKBACK, X_train.shape[-1]).to(DEVICE)
    out_dummy = model(x_dummy)
print(f"Smoke forward: input {tuple(x_dummy.shape)} → output {tuple(out_dummy.shape)}")

In [ ]:
# --- Entrenamiento TCN (salida binaria global) + smoke test ---
import random
import copy
import time
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader

# NB_TAG viene de la celda inicial (derivado del nombre del notebook)

BATCH_SIZE   = 64
LR           = 3e-4
EPOCHS_MAX   = 80
PATIENCE     = 12
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0

PIN_MEMORY  = (DEVICE.type == "cuda")
NUM_WORKERS = 0

POS_WEIGHT_T = torch.tensor([POS_WEIGHT]).float()

t_X_tr = torch.from_numpy(X_train).float()
t_y_tr = torch.from_numpy(y_train.astype(np.float32))
t_X_va = torch.from_numpy(X_val).float()
t_y_va = torch.from_numpy(y_val.astype(np.float32))
t_X_te = torch.from_numpy(X_test).float()
t_y_te = torch.from_numpy(y_test.astype(np.float32))


def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # reproducibilidad GPU (sin esto, las
    torch.backends.cudnn.benchmark     = False  # convoluciones cuDNN son no-deterministas)


def save_checkpoint(ckpt_path, epoch, model_state, val_auc_best, history, seed):
    torch.save({
        "epoch": epoch, "model_state": model_state, "val_auc_best": val_auc_best,
        "history": dict(history), "seed": seed,
        "config": {
            "NUM_CHANNELS": list(NUM_CHANNELS), "KERNEL_SIZE": KERNEL_SIZE,
            "DROPOUT": DROPOUT, "LOOKBACK": LOOKBACK,
            "PREDICTION_WINDOW": PREDICTION_WINDOW, "n_features": n_features,
        },
        "FEATURE_COLS": FEATURE_COLS, "mean_train": mean_train, "std_train": std_train,
    }, ckpt_path)


def evaluate(model, X_t, y_np):
    model.eval()
    with torch.no_grad():
        logits = model(X_t.to(DEVICE)).cpu().numpy()
    if y_np.sum() == 0 or y_np.sum() == len(y_np):
        return np.nan, logits
    return roc_auc_score(y_np, logits), logits


def train_one_seed(seed, verbose = True):
    set_all_seeds(seed)

    train_loader = DataLoader(
        TensorDataset(t_X_tr, t_y_tr), batch_size = BATCH_SIZE, shuffle = True,
        generator = torch.Generator().manual_seed(seed),
        num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY,
    )

    model = TCNClassifier(
        num_inputs = X_train.shape[-1], num_channels = NUM_CHANNELS,
        kernel_size = KERNEL_SIZE, dropout = DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr = LR, weight_decay = WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode = "max", factor = 0.5, patience = 5, min_lr = 1e-5,
    )
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight = POS_WEIGHT_T.to(DEVICE))

    history = {"train_loss": [], "val_auc": []}
    best_val_auc, best_state, patience_counter = -1.0, None, 0
    ckpt_path = os.path.join(RESULTS_DIR, f"{NB_TAG}_ckpt_seed{seed}.pt")

    for epoch in range(1, EPOCHS_MAX + 1):
        model.train()
        train_losses = []

        if verbose:
            pbar = tqdm(train_loader, desc = f"Epoch {epoch:>2}/{EPOCHS_MAX}", leave = True,
                        bar_format = "{desc}: {percentage:3.0f}%|{bar:25}| "
                                     "{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]{postfix}")
        else:
            pbar = train_loader

        for xb, yb in pbar:
            xb = xb.to(DEVICE, non_blocking = True)
            yb = yb.to(DEVICE, non_blocking = True)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_losses.append(loss.item())
            if verbose:
                pbar.set_postfix_str(f"loss={np.mean(train_losses):.4f}")

        if verbose:
            pbar.close()

        val_auc, _ = evaluate(model, t_X_va, y_val)
        scheduler.step(val_auc)

        history["train_loss"].append(np.mean(train_losses))
        history["val_auc"].append(val_auc)

        new_best = val_auc > best_val_auc
        if new_best:
            best_val_auc, best_state, patience_counter = val_auc, copy.deepcopy(model.state_dict()), 0
            save_checkpoint(ckpt_path, epoch, best_state, best_val_auc, history, seed)
        else:
            patience_counter += 1

        if verbose:
            mark = " *" if new_best else ""
            tqdm.write(f"           val_auc={val_auc:.4f}  best={best_val_auc:.4f}{mark}"
                       f"   lr={optimizer.param_groups[0]['lr']:.0e}")

        if patience_counter >= PATIENCE:
            if verbose:
                tqdm.write(f"Early stop tras epoch {epoch} (best val_auc={best_val_auc:.4f})")
            break

    model.load_state_dict(best_state)
    test_auc, test_logits = evaluate(model, t_X_te, y_test)

    return {"seed": seed, "history": history, "val_auc_best": best_val_auc,
            "test_auc": test_auc, "test_logits": test_logits, "ckpt_path": ckpt_path}


print(f"=== Configuración entrenamiento ({NB_TAG}) ===")
print(f"DEVICE={DEVICE}  BATCH_SIZE={BATCH_SIZE}  LR={LR}  EPOCHS_MAX={EPOCHS_MAX}  PATIENCE={PATIENCE}")
print(f"Loss: BCEWithLogits + pos_weight={POS_WEIGHT:.3f}  (target ratio positivos: {y_train.mean()*100:.1f}%)")
print()


## 5. Baselines y comparación

In [ ]:
# --- DIAGNÓSTICO: baselines (LogReg + persistencia naive sobre target global) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def aggregate(X):
    return np.concatenate([X.mean(axis = 1), X.std(axis = 1),
                           X[:, -1, :], X.max(axis = 1)], axis = 1)

X_tr_agg = aggregate(X_train)
X_va_agg = aggregate(X_val)
X_te_agg = aggregate(X_test)

# LogReg global (mismo target binario global que el TCN)
logreg = LogisticRegression(max_iter = 1000, class_weight = "balanced", random_state = SEED)
logreg.fit(X_tr_agg, y_train)
val_auc_lr  = roc_auc_score(y_val,  logreg.predict_proba(X_va_agg)[:, 1])
test_auc_lr = roc_auc_score(y_test, logreg.predict_proba(X_te_agg)[:, 1])
print(f"LogReg global:    val AUC = {val_auc_lr:.4f}   test AUC = {test_auc_lr:.4f}")

# Persistencia naive: mainshock en los últimos T días → predice 1
naive_score = np.zeros(len(y_test))
for i, t_idx in enumerate(np.where(test_mask)[0]):
    t_global = valid_t[t_idx]
    naive_score[i] = is_target_day[max(0, t_global - PREDICTION_WINDOW) : t_global].any()
naive_auc = roc_auc_score(y_test, naive_score)
print(f"Persistencia naive: test AUC = {naive_auc:.4f}")

In [ ]:
# --- Multi-seed del TCN (5 seeds) ---
SEEDS = [13, 42, 123, 256, 1024]
results_per_seed = []

print(f"=== Multi-seed loop del TCN {NB_TAG} ({len(SEEDS)} seeds) ===\n")
t0 = time.time()
for s_idx, s in enumerate(SEEDS, 1):
    print(f"--- Seed {s} ({s_idx}/{len(SEEDS)}) ---")
    res = train_one_seed(seed = s, verbose = False)
    results_per_seed.append(res)
    print(f"  val={res['val_auc_best']:.4f}   test={res['test_auc']:.4f}")
elapsed_ms = time.time() - t0

val_aucs  = np.array([r["val_auc_best"] for r in results_per_seed])
test_aucs = np.array([r["test_auc"]     for r in results_per_seed])

# Ensemble (media de logits sobre el test set)
logits_test_seeds = np.stack([r["test_logits"] for r in results_per_seed])  # (n_seeds, n_test)
logits_ensemble   = logits_test_seeds.mean(axis = 0)
ensemble_auc      = roc_auc_score(y_test, logits_ensemble)

print(f"\n=== Resumen multi-seed {NB_TAG} (n={len(SEEDS)}) ===")
print(f"Tiempo total: {elapsed_ms:.1f}s ({elapsed_ms/60:.1f} min)")
print()
print(f"Val  AUC: {val_aucs.mean():.4f} ± {val_aucs.std():.4f}   "
      f"rango=[{val_aucs.min():.4f}, {val_aucs.max():.4f}]")
print(f"Test AUC: {test_aucs.mean():.4f} ± {test_aucs.std():.4f}   "
      f"rango=[{test_aucs.min():.4f}, {test_aucs.max():.4f}]")
print(f"Ensemble: {ensemble_auc:.4f}")
print()
print(f"Referencias: LogReg=0.5447  TCN=0.5288  GNN=0.6953")

In [ ]:
# --- Comparación final: tabla + ROC + Wilcoxon + pickle ---
from sklearn.metrics import roc_curve
from scipy.stats import ttest_1samp
import pickle

print("=== Tabla comparativa (test set 2023–2024) ===")
print(f"{'Modelo':<35s}  {'Val AUC':>10s}  {'Test AUC':>10s}")
print("-" * 60)
print(f"{'Persistencia naive':<35s}  {'—':>10s}  {naive_auc:>10.4f}")
print(f"{'Regresion logistica {NB_TAG}':<35s}  {val_auc_lr:>10.4f}  {test_auc_lr:>10.4f}")
print(f"{'TCN {NB_TAG} multi-seed mean ± std':<35s}  {val_aucs.mean():>6.4f}±{val_aucs.std():.3f}  "
      f"{test_aucs.mean():>6.4f}±{test_aucs.std():.3f}")
print(f"{'TCN {NB_TAG} ensemble':<35s}  {'—':>10s}  {ensemble_auc:>10.4f}")
print()

# t-test TCN vs LogReg
t_stat, p_val = ttest_1samp(test_aucs, test_auc_lr)
direction = "supera" if test_aucs.mean() > test_auc_lr else "es inferior"
sig = " (p<0.05 SIGNIFICATIVO)" if p_val < 0.05 else " (no significativo)"
print(f"t-test TCN vs LogReg: t={t_stat:+.3f}, p={p_val:.4f} → TCN {direction} a LogReg{sig}")
print()

# Figura ROC
fig, ax = plt.subplots(figsize = (7, 7))
ax.plot([0, 1], [0, 1], color = "gray", linestyle = "--", linewidth = 1, label = "Aleatorio")

fpr, tpr, _ = roc_curve(y_test, naive_score)
ax.plot(fpr, tpr, linewidth = 1.5, label = f"Naive (AUC={naive_auc:.3f})")

fpr, tpr, _ = roc_curve(y_test, logreg.predict_proba(X_te_agg)[:, 1])
ax.plot(fpr, tpr, linewidth = 1.8, label = f"LogReg (AUC={test_auc_lr:.3f})")

for r in results_per_seed:
    fpr, tpr, _ = roc_curve(y_test, r["test_logits"])
    ax.plot(fpr, tpr, color = "gray", alpha = 0.35, linewidth = 0.8)

fpr, tpr, _ = roc_curve(y_test, logits_ensemble)
ax.plot(fpr, tpr, color = "darkred", linewidth = 2.2,
        label = f"TCN ensemble (AUC={ensemble_auc:.3f})")

ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC test 2023–2024 — TCN {NB_TAG} ({REGIONS_TAG}, T={PREDICTION_WINDOW}d)",
             fontweight = "bold")
ax.legend(loc = "lower right"); ax.grid(True, alpha = 0.3); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,
            f"{NB_TAG}_roc_test_T{PREDICTION_WINDOW}d.png"), dpi = 300, bbox_inches = "tight")
plt.show()

# Pickle
results_summary = {
    "config": {
        "REGIONS": REGIONS, "REGIONS_TAG": REGIONS_TAG, "STARTTIME": STARTTIME,
        "ENDTIME": ENDTIME, "MIN_MAG_CATALOGO": MIN_MAG_CATALOGO,
        "MIN_MAG_TARGET": MIN_MAG_TARGET, "PREDICTION_WINDOW": PREDICTION_WINDOW,
        "LOOKBACK": LOOKBACK, "FEATURE_COLS": FEATURE_COLS,
        "NUM_CHANNELS": list(NUM_CHANNELS), "KERNEL_SIZE": KERNEL_SIZE,
        "DROPOUT": DROPOUT, "SEEDS": SEEDS,
    },
    "y_test": y_test,
    "naive_auc": naive_auc, "naive_score": naive_score,
    "logreg": {"val_auc": val_auc_lr, "test_auc": test_auc_lr, "model": logreg},
    "tcn": {
        "history_per_seed": [r["history"] for r in results_per_seed],
        "val_aucs":         val_aucs,
        "test_aucs":        test_aucs,
        "test_logits_seeds": logits_test_seeds,
        "ensemble_auc":     ensemble_auc,
        "ensemble_logits":  logits_ensemble,
    },
    "ttest": {"t_stat": t_stat, "p_val": p_val},
}
pickle_path = os.path.join(RESULTS_DIR, f"{NB_TAG}_tcn_T{PREDICTION_WINDOW}d_results.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump(results_summary, f)
print(f"Resultados guardados en: {pickle_path}")

## 6. Evaluación final

In [ ]:
# --- Metricas completas (PR-AUC, Precision@K, Lift) + Molchan ---
# El objetivo es global (por día, no por celda), por lo que el baseline de Poisson
# sería una probabilidad constante (AUC 0.5, no discrimina sin dimensión espacial).
# Los baselines relevantes son la persistencia naive y la regresión logística.
from utils.evaluacion import metricas_completas, resumen_multiseed, plot_molchan

prob_logreg = logreg.predict_proba(X_te_agg)[:, 1]

metricas_por_seed = [metricas_completas(y_test, r["test_logits"]) for r in results_per_seed]
resumen      = resumen_multiseed(metricas_por_seed)
met_ensemble = metricas_completas(y_test, logits_ensemble)
met_logreg   = metricas_completas(y_test, prob_logreg)
met_naive    = metricas_completas(y_test, naive_score)

metricas_a_mostrar = ["roc_auc","pr_auc","precision@1%","recall@1%","lift@1%",
                      "precision@5%","recall@5%","lift@5%"]
print(f"{'Metrica':16} | {'TCN (media+/-std)':20} | {'ensemble':9} | {'LogReg':8} | {'naive':8}")
print("-" * 76)
for k in metricas_a_mostrar:
    r = resumen[k]
    print(f"{k:16} | {r['mean']:6.4f} +/- {r['std']:.4f}    | {met_ensemble[k]:>7.4f}  | "
          f"{met_logreg[k]:>7.4f} | {met_naive[k]:>7.4f}")
print(f"(base_rate test = {met_ensemble['base_rate']:.4f})")
print()
print("El objetivo es global (por dia, sin celdas), por lo que su AUC ya es")
print("      una AUC temporal pura -- no hay dimension espacial que infle el resultado.")
print("      La 'AUC temporal por celda' solo aplica a los notebooks espaciales 04a/05a.")

# Molchan: ensemble vs LogReg vs naive
fig, ax = plt.subplots(figsize=(6.5, 6.5))
plot_molchan(y_test, logits_ensemble, ax=ax, label=f"TCN {NB_TAG} (ensemble)", color="darkred")
plot_molchan(y_test, prob_logreg,     ax=ax, label="LogReg",                   color="steelblue")
plot_molchan(y_test, naive_score,     ax=ax, label="Persistencia naive",       color="gray")
ax.set_title(f"Diagrama de Molchan - {REGIONS_TAG} 2023-2024 (TCN global)", fontweight="bold")
plt.tight_layout()
molchan_path = os.path.join(FIGURES_DIR, f"{NB_TAG}_molchan_T{PREDICTION_WINDOW}d.png")
plt.savefig(molchan_path, dpi=300, bbox_inches="tight"); plt.show()
print(f"\nFigura Molchan: {molchan_path}")
